In [ ]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 07, Derived bracket

Companion notebook to [07_derived_bracket.md](07_derived_bracket.md). The construction `{a, b}_Q := [[a, Q]_base, b]_base`, Jacobi reduced to a single equation (`[Q, Q]_base = 0`), Koszul equivalence, and the Poisson / H-twisted Courant corners.

## The construction

`DerivedBracket(base, Q, degree_Q=...)`, pick a degree-1 `Q` on a Lie base. `|{·,·}_Q| = |Q| − 2 = −1`. Leibniz is universal; antisymmetry / Jacobi are conditional (flag=None).

In [ ]:
from jacopy.brackets.derived import DerivedBracket
from jacopy.brackets.lie import LieBracket
from jacopy.core.expr import Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry

reg = PropertyRegistry()
Q = Symbol('Q')
reg.declare(Q, Graded(degree=1))
lie = LieBracket()

d = DerivedBracket(lie, Q, degree_Q=1)
print('name   :', d.name)
print('degree :', d.degree)
print('antisym:', d.is_graded_antisymmetric,
      'leibniz:', d.satisfies_leibniz,
      'jacobi :', d.satisfies_graded_jacobi)

## Two-faced expansion

`expand`, both inner and outer base layers resolved; `expand_definition`, both layers kept as inert `BracketApply` nodes.

In [ ]:
a, b = Symbol('a'), Symbol('b')
for s in (a, b):
    reg.declare(s, Graded(degree=0))

print('expand           :', d.expand(a, b, reg))
print('expand_definition:', d.expand_definition(a, b, reg))

## Jacobi obstruction, three faces

A single condition: `[Q, Q]_base = 0`. On a Lie base it's trivial (`Q*Q − Q*Q`).

In [ ]:
print('expanded :', d.jacobi_obstruction(reg))
print('raw      :', d.jacobi_obstruction_raw())
cond = d.jacobi_condition(reg)
print('condition:', cond.name)
print('holds?   :', cond.holds(reg))

## `prove_jacobi`, `DerivedBracketStrategy`

Auto-dispatched on bracket type. Three steps: `DerivedBracketTheorem` → `base-bracket-expand` → `simplify`.

In [ ]:
from jacopy.proof.verifier import prove_jacobi

# a, b already declared Graded(0) above, add c.
c = Symbol('c')
reg.declare(c, Graded(degree=0))

chain = prove_jacobi(d, a, b, c, registry=reg)
print('chain len:', len(chain))
for st in chain.steps:
    print(' ', st.rule)
print('final:', chain.steps[-1].after)

## `acting_on`, Koszul equivalence

SN base + π generator + anchor ρ: `expand` auto-emits the Koszul three-term form. Structurally equal to `KoszulBracket(ρ).expand`.

In [ ]:
from jacopy.brackets.schouten import sn
from jacopy.brackets.koszul import KoszulBracket
from jacopy.calculus.anchor import Anchor

reg2 = PropertyRegistry()
pi = Symbol('π')
reg2.declare(pi, Graded(degree=1))
alpha, beta = Symbol('α'), Symbol('β')
for s in (alpha, beta):
    reg2.declare(s, Graded(degree=1))

rho = Anchor('ρ')
koszul_derived = DerivedBracket(sn, pi, degree_Q=1, acting_on=rho)
koszul_classical = KoszulBracket(rho)

lhs = koszul_derived.expand(alpha, beta)
rhs = koszul_classical.expand(alpha, beta)
print('derived :', lhs)
print('classic :', rhs)
print('equal?  :', lhs == rhs)

## Poisson-as-derived, the library wrapper

Mathematically `DerivedBracket(sn, π, degree_Q=1)` is the Poisson bracket. The generic `prove_jacobi` reduces the obstruction to `[·,·]_SN(π, π)` and surfaces it as an honest `ProofFailure`, the Poisson hypothesis `[π, π]_SN = 0` must be carried as an explicit assumption. `PoissonBracket.prove_jacobi_reduction` closes it in one step via the seeded theorem citation.

In [ ]:
from jacopy.library import theorem_book
from jacopy.library.declarations import Bivector, Functions
from jacopy.library.poisson import PoissonBracket

reg3 = PropertyRegistry()
pi3 = Bivector('π', registry=reg3)
f, g, h = Functions('f g h', degree=-1, registry=reg3)

poisson = PoissonBracket.from_bivector(pi3)
chain = poisson.prove_jacobi_reduction(f, g, h, registry=reg3)
print('reduction chain len:', len(chain))
print('  rule  :', chain.steps[0].rule)
print('  after :', chain.steps[0].after)

thm = theorem_book.get('poisson_jacobi')
print('theorem from_axioms:', thm.from_axioms)

## H-twist, the Courant conditional Jacobi

`CourantBracket(background_H=H)`: Jacobi ⟺ `dH = 0`. The untwisted default (`H=None`) is vacuous.

In [ ]:
from jacopy.brackets.courant import CourantBracket

reg4 = PropertyRegistry()
H = Symbol('H')
reg4.declare(H, Graded(degree=3))

print('untwisted:', CourantBracket().jacobi_condition(reg4).name)

C = CourantBracket(background_H=H)
print('twisted  :', C.is_twisted)
cond = C.jacobi_condition(reg4)
print('  name       :', cond.name)
print('  obstruction:', cond.obstruction)

## Next step

Stage D: the unified picture + foundations, [08_unified_picture.md](08_unified_picture.md).